# Business Entity Resolution -- full run notebook

Run every cell top to bottom, in order. Cells marked **EDIT ME** need a path/value changed before you run them -- everything else works as-is once the earlier cells have run.


## 0. Find your compressed dataset archives
Skip this cell if you already know the exact filenames/paths.


In [ ]:
%cd /nfsshare/sudharsan/ML-chall
!find . -maxdepth 3 -iname "*.zip" -o -iname "*.tar.gz" -o -iname "*.tgz" -o -iname "*.7z" -o -iname "*.rar" 2>/dev/null


## 1. Decompress both dataset archives
**EDIT ME**: replace the two filenames with whatever cell 0 found.
`decompress_dataset.py` must already be uploaded to this same folder (via MobaXterm's SFTP panel, or Jupyter's Upload button).


In [ ]:
!python3 decompress_dataset.py train_dataset_archive.ext --out dataset_raw_train
!python3 decompress_dataset.py test_dataset_archive.ext --out dataset_raw_test


## 2. Move the extracted files into the layout the pipeline expects
**EDIT ME** if step 1's FOUND paths differ from what's assumed here.


In [ ]:
!mkdir -p dataset/student_resource/dataset/train dataset/student_resource/dataset/test
!cp dataset_raw_train/train_source1.tsv dataset_raw_train/train_source2.tsv \
    dataset_raw_train/train_source3.tsv dataset_raw_train/train_ground_truth.tsv \
    dataset/student_resource/dataset/train/
!cp dataset_raw_test/test_source1.tsv dataset_raw_test/test_source2.tsv \
    dataset_raw_test/test_source3.tsv \
    dataset/student_resource/dataset/test/


## 3. Unzip the pipeline code
`business_entity_resolution_pipeline.zip` must already be uploaded to this folder.


In [ ]:
!rm -rf business_entity_resolution
!unzip -q business_entity_resolution_pipeline.zip
!ls business_entity_resolution/src


The listing above should show 13 files, including `postprocess.py`. If it doesn't, the zip you unzipped is an older version.


## 4. Confirm the kernel's Python, then install dependencies into it


In [ ]:
!python3 -c "import sys; print(sys.executable)"


In [ ]:
!pip install --user "pandas==2.0.3" "numpy==1.24.4" "scipy==1.10.1" "scikit-learn==1.3.2" rapidfuzz lightgbm


In [ ]:
!python3 -c "import pandas, numpy, sklearn, scipy, rapidfuzz, lightgbm; print('all imports OK')"


## 5. Confirm the real dataset landed correctly


In [ ]:
!ls dataset/student_resource/dataset/train
!ls dataset/student_resource/dataset/test


## 6. Train on the full real dataset (background -- this is the long step)


In [ ]:
!mkdir -p business_entity_resolution/model business_entity_resolution/logs
!nohup python3 business_entity_resolution/src/train.py \
    --train-dir dataset/student_resource/dataset/train \
    --model-out business_entity_resolution/model/matcher.pkl \
    > business_entity_resolution/logs/train.log 2>&1 &


Re-run the next cell as many times as you like to check progress -- it does not block.


In [ ]:
!tail -n 40 business_entity_resolution/logs/train.log


Wait for the log to show `[train] model saved to ...` before continuing.


## 7. Compare decision rules on val -- **send me this cell's output**


In [ ]:
!python3 business_entity_resolution/src/postprocess.py \
    --scored business_entity_resolution/model/val_scored.tsv --mode val


Note which rule wins, and (if it's an `expected_f05` variant) the lambda it printed -- both are needed in cell 9.


## 8. Run the trained model on the real test set


In [ ]:
!mkdir -p dataset/student_resource/output
!python3 business_entity_resolution/src/predict.py \
    --test-dir dataset/student_resource/dataset/test \
    --model business_entity_resolution/model/matcher.pkl \
    --out-dir dataset/student_resource/output


## 9. Apply the winning decision rule from step 7
**EDIT ME**: `--rule` and `--lambda-val` below are placeholders (`one_to_one_expected_f05` / `0.3`) -- replace with whatever cell 7 actually recommended. If a threshold-based rule won instead, use `--rule threshold --threshold 0.5` (or whatever threshold value performed best) and drop `--lambda-val`.


In [ ]:
!python3 business_entity_resolution/src/postprocess.py \
    --scored dataset/student_resource/output/scored_candidates.tsv \
    --mode test --rule one_to_one_expected_f05 --lambda-val 0.3 \
    --test-dir dataset/student_resource/dataset/test \
    --out-dir dataset/student_resource/output


## 10. Validate -- must print PASS before you upload anything


In [ ]:
!python3 dataset/student_resource/utils/validate_submission.py \
    --matching dataset/student_resource/output/matching_results.tsv \
    --candidate dataset/student_resource/output/candidate_pairs.tsv \
    --test-dir dataset/student_resource/dataset/test


If this prints `PASS`, upload `dataset/student_resource/output/matching_results.tsv` to the portal.
